Hyperspectral images of wheat flag leaves. 2410 images in total, 1590 for training (24 plots) and 820 for testing (12 different plots).

The goal is to predict 8 targets: GrainWeight, Gsw, PhiPS2 (continuous), Fertilizer (0.0, 0.5 or 1.0), and Heerup, Kvium, Rembrandt, Sheriff (one hot cultivar columns).

Pipeline: leaf segmentation in each image, valid pixel extraction, cross validation, final model training (Deep Sets with multi head attention pooling) and submission generation.


i: imports the libraries and sets the seed.

ii: defines the data paths.


In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tifffile as tiff

from scipy import ndimage
from scipy.signal import savgol_filter

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import KFold

seed = 0

np.random.seed(seed)
tf.random.set_seed(seed)

data_dir = "/kaggle/input/competitions/HyperLeaf2024"
work_dir = "/kaggle/working"
image_dir = os.path.join(data_dir, "images")

os.makedirs(work_dir, exist_ok=True)

print("data_dir :", data_dir)
print("image_dir:", image_dir)
print("work_dir :", work_dir)


i: stores the 204 central wavelengths of each band, value given by the competition

ii: used only to find the spectral axis of the cube (its position varies between files, so it is located by the length 204 instead of assuming a fixed position)


In [ ]:
wavelengths = np.array([
    397.32, 400.20, 403.09, 405.97, 408.85, 411.74, 414.63, 417.52, 420.40, 423.29, 426.19, 429.08,
    431.97, 434.87, 437.76, 440.66, 443.56, 446.45, 449.35, 452.25, 455.16, 458.06, 460.96, 463.87,
    466.77, 469.68, 472.59, 475.50, 478.41, 481.32, 484.23, 487.14, 490.06, 492.97, 495.89, 498.80,
    501.72, 504.64, 507.56, 510.48, 513.40, 516.33, 519.25, 522.18, 525.10, 528.03, 530.96, 533.89,
    536.82, 539.75, 542.68, 545.62, 548.55, 551.49, 554.43, 557.36, 560.30, 563.24, 566.18, 569.12,
    572.07, 575.01, 577.96, 580.90, 583.85, 586.80, 589.75, 592.70, 595.65, 598.60, 601.55, 604.51,
    607.46, 610.42, 613.38, 616.34, 619.30, 622.26, 625.22, 628.18, 631.15, 634.11, 637.08, 640.04,
    643.01, 645.98, 648.95, 651.92, 654.89, 657.87, 660.84, 663.81, 666.79, 669.77, 672.75, 675.73,
    678.71, 681.69, 684.67, 687.65, 690.64, 693.62, 696.61, 699.60, 702.58, 705.57, 708.57, 711.56,
    714.55, 717.54, 720.54, 723.53, 726.53, 729.53, 732.53, 735.53, 738.53, 741.53, 744.53, 747.54,
    750.54, 753.55, 756.56, 759.56, 762.57, 765.58, 768.60, 771.61, 774.62, 777.64, 780.65, 783.67,
    786.68, 789.70, 792.72, 795.74, 798.77, 801.79, 804.81, 807.84, 810.86, 813.89, 816.92, 819.95,
    822.98, 826.01, 829.04, 832.07, 835.11, 838.14, 841.18, 844.22, 847.25, 850.29, 853.33, 856.37,
    859.42, 862.46, 865.50, 868.55, 871.60, 874.64, 877.69, 880.74, 883.79, 886.84, 889.90, 892.95,
    896.01, 899.06, 902.12, 905.18, 908.24, 911.30, 914.36, 917.42, 920.48, 923.55, 926.61, 929.68,
    932.74, 935.81, 938.88, 941.95, 945.02, 948.10, 951.17, 954.24, 957.32, 960.40, 963.47, 966.55,
    969.63, 972.71, 975.79, 978.88, 981.96, 985.05, 988.13, 991.22, 994.31, 997.40, 1000.49, 1003.58,
])

print("Number of bands:", len(wavelengths))


i: load_cube_hwc reads the tiff and moves the spectral axis to the last position, becomes (height, width, 204)

ii: segment_leaf marks a pixel as leaf when the sum of the 204 bands is greater than 0 (the background comes out exactly zero in every band), removes small connected components (the leaf typically appears in 2 pieces, so it keeps every large component, not just the biggest one), erodes the final mask by erosion_iterations steps to remove mixed pixels at the border (keeps the original mask if the erosion zeroes everything out) and returns the leaf bounding box with 2px of margin

iii: compute_axis_regions finds the leaf principal axis (PCA on the pixel coordinates), orients it consistently across images (the tip is always the narrower side, measured by the perpendicular spread at both ends), and splits the pixels into n_regions bands of equal width along this axis, from the tip (region 0) to the base (region n_regions-1)

In [ ]:
min_component_fraction = 0.01  # minimum connected component size to count as leaf (fraction of image area). in the samples the real components sit at 5-20%, so 1% already leaves margin against noise.
bbox_padding = 2  # margin in pixels around the leaf in the crop.
erosion_iterations = 0  # binary erosion steps on the final mask, each step removes ~1px from the border (reduces mixed leaf/background pixels).
n_regions = 5  # number of bands along the leaf tip-to-base axis.


def load_cube_hwc(image_path, wavelengths):
    cube = tiff.imread(image_path)
    spectral_axes = [ax for ax, size in enumerate(cube.shape) if size == len(wavelengths)]
    if len(spectral_axes) != 1:
        raise ValueError(f"Could not identify the spectral axis of {image_path}: shape={cube.shape}")
    cube = np.moveaxis(cube, spectral_axes[0], -1)
    return cube.astype(np.float32)


def segment_leaf(cube_hwc):
    total = cube_hwc.sum(axis=-1)
    zero_mask = total > 0

    labeled, n_components = ndimage.label(zero_mask)
    if n_components == 0:
        mask = zero_mask
        n_kept = 0
    else:
        sizes = ndimage.sum(zero_mask, labeled, index=np.arange(1, n_components + 1))
        min_size = min_component_fraction * zero_mask.size
        keep_labels = np.where(sizes >= min_size)[0] + 1
        mask = np.isin(labeled, keep_labels)
        n_kept = len(keep_labels)

    if erosion_iterations > 0:
        eroded_mask = ndimage.binary_erosion(mask, iterations=erosion_iterations)
        if eroded_mask.any():
            mask = eroded_mask

    ys, xs = np.where(mask)
    if ys.size == 0:
        bbox = (0, cube_hwc.shape[0], 0, cube_hwc.shape[1])
    else:
        r0 = max(int(ys.min()) - bbox_padding, 0)
        r1 = min(int(ys.max()) + bbox_padding + 1, cube_hwc.shape[0])
        c0 = max(int(xs.min()) - bbox_padding, 0)
        c1 = min(int(xs.max()) + bbox_padding + 1, cube_hwc.shape[1])
        bbox = (r0, r1, c0, c1)

    qc = {
        "leaf_pixel_fraction": float(mask.mean()),
        "n_raw_components": int(n_components),
        "n_kept_components": int(n_kept),
    }
    return mask, bbox, qc


def compute_axis_regions(cropped_mask, n_regions):
    ys, xs = np.where(cropped_mask)
    coords = np.stack([ys, xs], axis=1).astype(np.float64)
    centroid = coords.mean(axis=0)
    centered = coords - centroid

    cov = np.cov(centered, rowvar=False)
    eigvals, eigvecs = np.linalg.eigh(cov)
    principal_axis = eigvecs[:, np.argmax(eigvals)]
    perpendicular_axis = eigvecs[:, np.argmin(eigvals)]

    axis_pos = centered @ principal_axis
    axis_perp = centered @ perpendicular_axis

    order = np.argsort(axis_pos)
    edge = max(1, len(order) // 10)
    width_low = np.std(axis_perp[order[:edge]])
    width_high = np.std(axis_perp[order[-edge:]])
    if width_low > width_high:
        axis_pos = -axis_pos

    axis_range = axis_pos.max() - axis_pos.min()
    axis_pos_norm = (axis_pos - axis_pos.min()) / (axis_range + 1e-8)
    region_idx = np.clip((axis_pos_norm * n_regions).astype(np.int32), 0, n_regions - 1)
    return region_idx.astype(np.int8)


i: runs the segmentation on every image in image_dir and builds the audit table (qc_df), the array with all valid pixels (pixels_all) and the parallel array with the region of each pixel (regions_all, 0=tip to n_regions-1=base)

ii: each pixel goes through the 1st spectral derivative (Savitzky-Golay), replacing the raw value with the rate of change between neighboring bands. this removes additive and multiplicative baseline shift between images, preserving the position of the absorption peaks

In [ ]:
image_files = sorted(f for f in os.listdir(image_dir) if f.lower().endswith(".tiff"))
print(f"Found {len(image_files)} images in {image_dir}")

savgol_window = 11    # window of the 1st Savitzky-Golay derivative, in number of bands (~30nm).
savgol_polyorder = 2  # degree of the polynomial fit inside the window.
band_spacing = float(np.mean(np.diff(wavelengths)))

pixel_blocks = []
region_blocks = []
qc_rows = []
offset = 0

for filename in image_files:
    image_id = os.path.splitext(filename)[0]
    image_path = os.path.join(image_dir, filename)

    cube = load_cube_hwc(image_path, wavelengths)
    mask, bbox, qc = segment_leaf(cube)
    r0, r1, c0, c1 = bbox

    cropped_cube = cube[r0:r1, c0:c1]
    cropped_mask = mask[r0:r1, c0:c1]
    pixels = cropped_cube[cropped_mask]
    pixels = savgol_filter(pixels, window_length=savgol_window, polyorder=savgol_polyorder, deriv=1, delta=band_spacing, axis=1)
    regions = compute_axis_regions(cropped_mask, n_regions)
    pixel_blocks.append(pixels)
    region_blocks.append(regions)

    qc_rows.append({
        "ImageId": image_id,
        "n_pixels": pixels.shape[0],
        "leaf_pixel_fraction": qc["leaf_pixel_fraction"],
        "n_raw_components": qc["n_raw_components"],
        "n_kept_components": qc["n_kept_components"],
        "bbox_r0": r0, "bbox_r1": r1, "bbox_c0": c0, "bbox_c1": c1,
        "start_idx": offset,
        "end_idx": offset + pixels.shape[0],
    })
    offset += pixels.shape[0]

qc_df = pd.DataFrame(qc_rows)
pixels_all = np.concatenate(pixel_blocks, axis=0).astype(np.float32)
regions_all = np.concatenate(region_blocks, axis=0).astype(np.int8)

n_empty = (qc_df["n_pixels"] == 0).sum()
print("pixels_all shape:", pixels_all.shape)
print("regions_all shape:", regions_all.shape)
print(f"Images with no leaf pixels at all: {n_empty}")

qc_df


i: plots the total intensity, the final mask and the crop (bounding box) of a few images, to visually check that the segmentation is correct

ii: uses the real segment_leaf function, the same code that runs on the whole dataset


In [ ]:
sample_files = image_files[:4]
fig, axes = plt.subplots(len(sample_files), 3, figsize=(12, 3 * len(sample_files)))

for row, filename in enumerate(sample_files):
    cube = load_cube_hwc(os.path.join(image_dir, filename), wavelengths)
    mask, bbox, qc = segment_leaf(cube)
    total = cube.sum(axis=-1)
    r0, r1, c0, c1 = bbox

    axes[row, 0].imshow(total, cmap="inferno")
    axes[row, 0].set_title(f"{filename}\ntotal intensity")
    axes[row, 0].axis("off")

    axes[row, 1].imshow(mask, cmap="gray")
    axes[row, 1].set_title(f"final mask ({qc['leaf_pixel_fraction']*100:.1f}% of image)")
    axes[row, 1].axis("off")

    axes[row, 2].imshow(total[r0:r1, c0:c1], cmap="inferno")
    axes[row, 2].set_title("crop (bounding box)")
    axes[row, 2].axis("off")

plt.tight_layout()
plt.show()


i: picks 3 random images (fixed by the seed, reproducible) and draws the n_regions discrete regions (tip to base) overlaid on the leaf, using compute_axis_regions

In [ ]:
rng_viz = np.random.default_rng(seed)
sample_files = rng_viz.choice(image_files, size=3, replace=False)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for ax, filename in zip(axes, sample_files):
    cube = load_cube_hwc(os.path.join(image_dir, filename), wavelengths)
    mask, bbox, _ = segment_leaf(cube)
    r0, r1, c0, c1 = bbox
    cropped_mask = mask[r0:r1, c0:c1]

    ys, xs = np.where(cropped_mask)
    regions = compute_axis_regions(cropped_mask, n_regions)

    region_map = np.full(cropped_mask.shape, np.nan, dtype=np.float32)
    region_map[ys, xs] = regions
    region_map_masked = np.ma.masked_invalid(region_map)

    tip_mask = regions == 0
    base_mask = regions == n_regions - 1
    tip_point = (ys[tip_mask].mean(), xs[tip_mask].mean())
    base_point = (ys[base_mask].mean(), xs[base_mask].mean())

    im = ax.imshow(region_map_masked, cmap="viridis", vmin=0, vmax=n_regions - 1)
    ax.annotate(
        "", xy=(tip_point[1], tip_point[0]), xytext=(base_point[1], base_point[0]),
        arrowprops=dict(arrowstyle="->", color="red", linewidth=2),
    )
    ax.set_title(f"{filename}\n(purple=tip, yellow=base)")
    ax.axis("off")

fig.colorbar(im, ax=axes, ticks=range(n_regions), label="region", shrink=0.7)
plt.show()


i: saves pixels_all, regions_all and qc_df to disk (leaf_pixels.npy, leaf_pixel_regions.npy and leaf_pixels_index.csv)

ii: reloads with mmap_mode="r" so everything does not need to fit in ram at once

In [ ]:
pixels_path = os.path.join(work_dir, "leaf_pixels.npy")
regions_path = os.path.join(work_dir, "leaf_pixel_regions.npy")
index_path = os.path.join(work_dir, "leaf_pixels_index.csv")

np.save(pixels_path, pixels_all)
np.save(regions_path, regions_all)
qc_df.to_csv(index_path, index=False)

print("Saved:")
print(" -", pixels_path, pixels_all.shape, pixels_all.dtype)
print(" -", regions_path, regions_all.shape, regions_all.dtype)
print(" -", index_path)

pixels_mmap = np.load(pixels_path, mmap_mode="r")
regions_mmap = np.load(regions_path, mmap_mode="r")
qc_df_loaded = pd.read_csv(index_path, dtype={"ImageId": str})
print("Reload OK:", pixels_mmap.shape, regions_mmap.shape)


i: joins the audit table with the train.csv labels by ImageId

ii: splits who has a label (labeled_index, used for training) from who does not (unlabeled_index, which is the competition's own test set)


In [ ]:
regression_columns = ["GrainWeight", "Gsw", "PhiPS2"]
cultivar_columns = ["Heerup", "Kvium", "Rembrandt", "Sheriff"]
fertilizer_classes = np.array([0.0, 0.5, 1.0])

train_csv_path = os.path.join(data_dir, "train.csv")
train_labels = pd.read_csv(train_csv_path, dtype={"ImageId": str})

labeled_index = qc_df_loaded.merge(train_labels, on="ImageId", how="inner")
unlabeled_index = qc_df_loaded[~qc_df_loaded["ImageId"].isin(train_labels["ImageId"])].reset_index(drop=True)

print(f"{len(labeled_index)} labeled images / {len(unlabeled_index)} unlabeled images")
print(f"the full train.csv has {len(train_labels)} rows (official training dataset)")
labeled_index.head()


i: implements the competition's official metric (evaluation.JPG). mean squared error per target, divided by the target variance (standardizes scale across targets of different magnitudes) and weighted, with the 4 cultivar columns summed weighing the same as a single target (0.25 each)

ii: sigma comes from the full train.csv (1590 official rows), not just the reduced sample

iii: the formula in the image does not explicitly show the division by the sum of the weights, but it only matches the "~1.0 when guessing the train mean" mentioned in the documentation if divided by that sum. implemented this way, the cross validation cell compares the model against this baseline

In [ ]:
all_targets = regression_columns + ["Fertilizer"] + cultivar_columns
target_weights = np.array([1.0, 1.0, 1.0, 1.0, 0.25, 0.25, 0.25, 0.25])
target_sigma = train_labels[all_targets].std(ddof=0).values
global_mean = train_labels[all_targets].mean().values  # baseline (predicting the train mean), used as a reference in cross validation


def competition_score(y_true, y_pred):
    mse_per_target = ((np.asarray(y_true) - np.asarray(y_pred)) ** 2).mean(axis=0)
    standardized = mse_per_target / (target_sigma ** 2)
    return float((target_weights * standardized).sum() / target_weights.sum())


print("Weights per target:", dict(zip(all_targets, target_weights)))
print("Standard deviation in the full train.csv:", {k: round(v, 4) for k, v in zip(all_targets, target_sigma)})


i: PixelSetSequence builds the training batches: every epoch it samples k_per_region[r] pixels from each region r of each image (with replacement if the region has fewer pixels than that, or using every pixel of the image if the region comes up empty), to train in fixed size batches without resizing any image

ii: sends leaf_area (log of the number of valid pixels) along, so the model does not lose the notion of leaf size

In [ ]:
class PixelSetSequence(keras.utils.Sequence):
    def __init__(self, index_df, pixels_mmap, regions_mmap, n_regions, k_per_region, batch_size=16,
                 regression_scaler=None, shuffle=True, seed=seed):
        super().__init__()
        self.index_df = index_df.reset_index(drop=True)
        self.pixels_mmap = pixels_mmap
        self.regions_mmap = regions_mmap
        self.n_regions = n_regions
        self.k_per_region = np.asarray(k_per_region)
        self.batch_size = batch_size
        self.shuffle = shuffle
        self.rng = np.random.default_rng(seed)

        if regression_scaler is None:
            regression_scaler = StandardScaler().fit(self.index_df[regression_columns].values)
        self.regression_scaler = regression_scaler

    def __len__(self):
        return int(np.ceil(len(self.index_df) / self.batch_size))

    def on_epoch_end(self):
        if self.shuffle:
            self.index_df = self.index_df.sample(frac=1, random_state=int(self.rng.integers(1_000_000))).reset_index(drop=True)

    def __getitem__(self, batch_idx):
        rows = self.index_df.iloc[batch_idx * self.batch_size:(batch_idx + 1) * self.batch_size]

        n_features = self.pixels_mmap.shape[1]
        region_batches = [np.zeros((len(rows), int(self.k_per_region[r]), n_features), dtype=np.float32) for r in range(self.n_regions)]
        area_batch = np.zeros((len(rows), 1), dtype=np.float32)

        for i, row in enumerate(rows.itertuples()):
            row_regions = np.asarray(self.regions_mmap[row.start_idx:row.end_idx])
            for r in range(self.n_regions):
                k_r = int(self.k_per_region[r])
                available = np.where(row_regions == r)[0] + row.start_idx
                if len(available) == 0:
                    available = np.arange(row.start_idx, row.end_idx)
                replace = len(available) < k_r
                chosen = self.rng.choice(available, size=k_r, replace=replace)
                region_batches[r][i] = self.pixels_mmap[chosen]
            area_batch[i, 0] = np.log1p(row.n_pixels)

        regression_batch = self.regression_scaler.transform(rows[regression_columns].values).astype(np.float32)
        fertilizer_batch = (rows["Fertilizer"].values[:, None] == fertilizer_classes[None, :]).astype(np.float32)
        cultivar_batch = rows[cultivar_columns].values.astype(np.float32)

        x = {f"pixel_set_region_{r}": region_batches[r] for r in range(self.n_regions)}
        x["leaf_area"] = area_batch
        y = {"regression": regression_batch, "fertilizer": fertilizer_batch, "cultivar": cultivar_batch}
        return x, y


i: this is a Deep Sets style architecture. each leaf is treated as an unordered set of pixel spectra (not an image), so the model needs an aggregation step that is invariant to pixel order and works with a variable number of pixels per leaf

ii: AttentionPooling is the multi head attention pooling layer that does this aggregation. instead of a fixed pooling rule (mean or max), it uses a single learned query vector (the "seed") that attends over all the pixels in a region. the attention mechanism computes, for every pixel, how relevant it is to this seed, and returns a weighted combination of the pixels as the region summary. training decides which pixels matter most, it is not hand designed

iii: build_pixel_set_model takes one pixel set input per region (pixel_set_region_0 to pixel_set_region_{n_regions-1}), each with 204 spectral bands. a small per pixel MLP (pixel_hidden) is applied first, shared across every region (same weights), turning each raw pixel spectrum into a learned embedding. each region then has its own AttentionPooling instance (separate weights per region), since the tip and the base of the leaf are expected to carry different information and deserve their own attention mechanism. the n_regions resulting summary vectors are concatenated with leaf_area (a scalar feature, the log of the leaf pixel count) before the trunk

iv: the trunk is a small shared MLP on top of the concatenated region summaries, followed by 3 output heads: a linear regression head (GrainWeight, Gsw, PhiPS2) and two softmax classification heads (Fertilizer, Cultivar)

v: raw pixels are uint16 counts up to 65535, and without normalizing the bands the gradient explodes in the 1st epoch. the normalizer is passed in already fitted from outside, adapted on the extracted pixels themselves

In [ ]:
class AttentionPooling(layers.Layer):
    def __init__(self, num_heads=4, key_dim=16, **kwargs):
        super().__init__(**kwargs)
        self.num_heads = num_heads
        self.key_dim = key_dim

    def build(self, input_shape):
        embed_dim = input_shape[-1]
        self.seed = self.add_weight(
            shape=(1, 1, embed_dim), initializer="glorot_uniform", trainable=True, name="seed"
        )
        self.attention = layers.MultiHeadAttention(num_heads=self.num_heads, key_dim=self.key_dim)
        super().build(input_shape)

    def call(self, x):
        batch_size = tf.shape(x)[0]
        seed = tf.tile(self.seed, [batch_size, 1, 1])
        pooled = self.attention(query=seed, value=x, key=x)
        return tf.squeeze(pooled, axis=1)


def build_pixel_set_model(
    n_bands,
    pixel_normalizer,
    n_regions,
    pixel_hidden=(128, 64),   # per pixel dense layer sizes, shared across regions
    trunk_hidden=(128, 64),   # dense layer sizes after pooling
    dropout=0.15,             # dropout rate in the trunk
    attention_heads=4,        # attention heads per region pooling
    attention_key_dim=16,     # dimension of each attention head
):
    region_inputs = [keras.Input(shape=(None, n_bands), name=f"pixel_set_region_{r}") for r in range(n_regions)]
    area_input = keras.Input(shape=(1,), name="leaf_area")

    pixel_dense_layers = [layers.Dense(units, activation="relu") for units in pixel_hidden]

    pooled_regions = []
    for r, region_input in enumerate(region_inputs):
        x = pixel_normalizer(region_input)
        for dense in pixel_dense_layers:
            x = dense(x)
        pooled = AttentionPooling(num_heads=attention_heads, key_dim=attention_key_dim, name=f"attention_pooling_region_{r}")(x)
        pooled_regions.append(pooled)

    pooled = layers.Concatenate()(pooled_regions + [area_input])

    trunk = pooled
    for units in trunk_hidden:
        trunk = layers.Dense(units, activation="relu")(trunk)
        trunk = layers.Dropout(dropout)(trunk)

    regression_out = layers.Dense(len(regression_columns), activation="linear", name="regression")(trunk)
    fertilizer_out = layers.Dense(len(fertilizer_classes), activation="softmax", name="fertilizer")(trunk)
    cultivar_out = layers.Dense(len(cultivar_columns), activation="softmax", name="cultivar")(trunk)

    return keras.Model(
        inputs=region_inputs + [area_input],
        outputs=[regression_out, fertilizer_out, cultivar_out],
        name="pixel_set_regressor",
    )


def compile_new_model(pixel_normalizer, n_regions):
    m = build_pixel_set_model(n_bands=len(wavelengths), pixel_normalizer=pixel_normalizer, n_regions=n_regions)
    m.compile(
        optimizer=keras.optimizers.Adam(learning_rate=1e-3),
        loss={
            "regression": "mse",
            "fertilizer": "categorical_crossentropy",
            "cultivar": "categorical_crossentropy",
        },
        loss_weights={
            "regression": 1.0,  # regression already covers 3 targets (GrainWeight+Gsw+PhiPS2), which together are worth 3 weight units in the official metric
            "fertilizer": 0.3,  # fertilizer is worth 1 weight unit in the metric (against 3 for regression)
            "cultivar": 0.3,    # the 4 cultivar columns together are also worth 1 weight unit in the metric (0.25 each, same logic as fertilizer above)
        },
    )
    return m


i: decode_predictions undoes the prediction scaling: regression goes back to the original unit via the scaler, fertilizer becomes the expected value of the softmax distribution (probs times [0.0, 0.5, 1.0]), cultivar keeps its own softmax probabilities

ii: does not apply argmax or rounding to either classification

iii: predict_for_images uses decode_predictions and runs the model image by image, splitting that image's valid pixels by region and passing every pixel of each region (no sampling, no resizing) to the corresponding model input. the fixed k_per_region is only needed to pack training batches

In [ ]:
def decode_predictions(regression_scaled, fertilizer_probs, cultivar_probs, regression_scaler):
    regression = regression_scaler.inverse_transform(regression_scaled)
    fertilizer = fertilizer_probs @ fertilizer_classes
    cultivar = cultivar_probs
    return regression, fertilizer, cultivar


def predict_for_images(model, index_df, pixels_mmap, regions_mmap, n_regions, regression_scaler):
    rows_out = []
    for row in index_df.itertuples():
        row_regions = np.asarray(regions_mmap[row.start_idx:row.end_idx])
        row_pixels = np.asarray(pixels_mmap[row.start_idx:row.end_idx])

        x = {}
        for r in range(n_regions):
            region_mask = row_regions == r
            region_pixels = row_pixels[region_mask] if region_mask.any() else row_pixels
            x[f"pixel_set_region_{r}"] = region_pixels[None, :, :]
        x["leaf_area"] = np.array([[np.log1p(row.n_pixels)]], dtype=np.float32)

        reg_scaled, fert_probs, cult_probs = model.predict(x, verbose=0)
        reg, fert, cult = decode_predictions(reg_scaled, fert_probs, cult_probs, regression_scaler)

        rows_out.append({
            "ImageId": row.ImageId,
            "GrainWeight": reg[0, 0],
            "Gsw": reg[0, 1],
            "PhiPS2": reg[0, 2],
            "Fertilizer": fert[0],
            "Heerup": cult[0, 0],
            "Kvium": cult[0, 1],
            "Rembrandt": cult[0, 2],
            "Sheriff": cult[0, 3],
        })
    return pd.DataFrame(rows_out)


i: defines the training hyperparameters, used both in cross validation and in the final model (epochs_final is defined later, inside cross validation itself)

ii: k_per_region is no longer a single number, it is an array with one value per region, proportional to the average fraction of pixels each region occupies in the whole dataset (larger regions like the base get more sample pixels than smaller regions like the tip), with a floor so no region ends up too small for attention to make sense

In [ ]:
k_total = 2500  # total pixel budget per image at each training step, split across the n_regions regions proportional to the average size of each one in the dataset
min_k_per_region = 200  # minimum pixel floor per region, so small regions (typically the tip) do not end up with too few pixels for attention to make sense
batch_size = 32

region_fractions = np.array([(np.asarray(regions_mmap) == r).mean() for r in range(n_regions)])
k_per_region = np.maximum(np.round(k_total * region_fractions), min_k_per_region).astype(int)

print("Average pixel fraction per region (tip to base):", np.round(region_fractions, 3))
print("k_per_region per region:", k_per_region)


i: cross validation using the competition's official metric, with early stopping monitoring val_loss on each fold

ii: the best epoch of each fold is averaged into epochs_final, used later in the final model. the result is compared against the baseline of predicting the train mean

In [ ]:
n_folds = 5
max_epochs_cv = 150
patience = 15

kf = KFold(n_splits=n_folds, shuffle=True, random_state=seed)
fold_scores = []
fold_best_epochs = []

for fold_idx, (train_idx, val_idx) in enumerate(kf.split(labeled_index)):
    train_fold = labeled_index.iloc[train_idx].reset_index(drop=True)
    val_fold = labeled_index.iloc[val_idx].reset_index(drop=True)

    keras.backend.clear_session()
    tf.random.set_seed(seed)
    fold_normalizer = layers.Normalization(axis=-1, name="pixel_normalizer_fold")
    fold_normalizer.adapt(np.asarray(pixels_mmap))
    fold_model = compile_new_model(fold_normalizer, n_regions=n_regions)

    fold_train_sequence = PixelSetSequence(
        train_fold, pixels_mmap, regions_mmap, n_regions=n_regions,
        k_per_region=k_per_region, batch_size=batch_size, seed=seed,
    )
    fold_val_sequence = PixelSetSequence(
        val_fold, pixels_mmap, regions_mmap, n_regions=n_regions,
        k_per_region=k_per_region, batch_size=batch_size, seed=seed, shuffle=False,
        regression_scaler=fold_train_sequence.regression_scaler,
    )

    early_stopping = keras.callbacks.EarlyStopping(monitor="val_loss", patience=patience, restore_best_weights=True)
    history = fold_model.fit(
        fold_train_sequence, validation_data=fold_val_sequence,
        epochs=max_epochs_cv, callbacks=[early_stopping], verbose=0,
    )
    best_epoch = int(np.argmin(history.history["val_loss"])) + 1
    fold_best_epochs.append(best_epoch)

    fold_preds = predict_for_images(fold_model, val_fold, pixels_mmap, regions_mmap, n_regions, fold_train_sequence.regression_scaler)
    fold_score = competition_score(val_fold[all_targets].values, fold_preds[all_targets].values)
    fold_scores.append(fold_score)
    print(f"Fold {fold_idx + 1}/{n_folds}: score {fold_score:.4f}, best epoch {best_epoch}")

fold_scores = np.array(fold_scores)
epochs_final = int(round(np.mean(fold_best_epochs)))
baseline_score = competition_score(labeled_index[all_targets].values, np.tile(global_mean, (len(labeled_index), 1)))

print(f"CV score: mean {fold_scores.mean():.4f}, std {fold_scores.std():.4f}")
print(f"Baseline (train mean): {baseline_score:.4f}")
print(f"number of epochs chosen as the average of the best epoch per fold: {epochs_final}")


i: trains one full model per seed in ensemble_seeds, each from scratch, with keras.backend.clear_session() before each seed to free memory (the normalizer is also recreated each time)

ii: stores each seed's prediction in ensemble_predictions, using the epochs_final value from cross validation

In [ ]:
ensemble_seeds = [1, 2, 3, 4, 5]
ensemble_predictions = []
for ensemble_seed in ensemble_seeds:
    keras.backend.clear_session()
    tf.random.set_seed(ensemble_seed)
    final_normalizer = layers.Normalization(axis=-1, name="pixel_normalizer_final")
    final_normalizer.adapt(np.asarray(pixels_mmap))
    model = compile_new_model(final_normalizer, n_regions=n_regions)
    final_train_sequence = PixelSetSequence(
        labeled_index, pixels_mmap, regions_mmap, n_regions=n_regions,
        k_per_region=k_per_region, batch_size=batch_size, seed=ensemble_seed,
    )
    history = model.fit(final_train_sequence, epochs=epochs_final, verbose=2)
    preds = predict_for_images(model, unlabeled_index, pixels_mmap, regions_mmap, n_regions, final_train_sequence.regression_scaler)
    ensemble_predictions.append(preds.set_index("ImageId"))

model.summary()


i: takes the simple average of every ensemble seed's predictions (ensemble_predictions), column by column, and returns with ImageId as a regular column

In [ ]:
submission_df = (sum(ensemble_predictions) / len(ensemble_predictions)).reset_index()
submission_df


i: checks that the prediction columns match sample_submission.csv exactly, same name and same order

ii: checks how many ImageId from sample_submission.csv were covered by the prediction, and saves submission.csv in work_dir


In [ ]:
sample_submission_path = os.path.join(data_dir, "sample_submission.csv")
sample_submission = pd.read_csv(sample_submission_path, dtype={"ImageId": str})

submission_df = submission_df[sample_submission.columns.tolist()]
assert list(submission_df.columns) == list(sample_submission.columns), "columns do not match sample_submission.csv"
print("Columns match sample_submission.csv:", list(submission_df.columns))

covered = set(submission_df["ImageId"]) & set(sample_submission["ImageId"])
print(f"{len(covered)} of {len(sample_submission)} ImageId from sample_submission.csv covered by the prediction")

submission_path = os.path.join(work_dir, "submission.csv")
submission_df.to_csv(submission_path, index=False)
print("Saved:", submission_path)

submission_df


i: unlabeled_index has the 820 test images, submission.csv comes out with all of them

In [ ]:
solution_path = os.path.join(data_dir, "solution.csv")
solution_df = pd.read_csv(solution_path, dtype={"ImageId": str})

submission_check_df = pd.read_csv(os.path.join(work_dir, "submission.csv"), dtype={"ImageId": str})

merged = solution_df.merge(submission_check_df, on="ImageId", suffixes=("_true", "_pred"))

true_cols = [f"{c}_true" for c in all_targets]
pred_cols = [f"{c}_pred" for c in all_targets]


def score_subset(df, mask=None):
    subset = df if mask is None else df[mask]
    return competition_score(subset[true_cols].values, subset[pred_cols].values), len(subset)


score_all, n_all = score_subset(merged)
score_public, n_public = score_subset(merged, merged["Usage"] == "Public")
score_private, n_private = score_subset(merged, merged["Usage"] == "Private")

baseline_test_pred = np.tile(global_mean, (len(merged), 1))
baseline_test_score = competition_score(merged[true_cols].values, baseline_test_pred)

print(f"Real score (all {n_all} test images): {score_all:.4f}")
print(f"Real score (public, {n_public} images): {score_public:.4f}")
print(f"Real score (private, {n_private} images): {score_private:.4f}")
print(f"Baseline on the real test set (predict train mean): {baseline_test_score:.4f}")
